# LM Conversacional Basico

Fine-tuning b?sico de un modelo causal peque?o con Transformers y `daily_dialog`.

## Instalaci?n

Ejecuta la instalaci?n solo si tu entorno no tiene las dependencias.

In [ ]:
# %pip install -r requirements.txt

## Carga de modelo y dataset

In [ ]:
import os
os.environ.setdefault("WANDB_DISABLED", "true")

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

raw_dataset = load_dataset("daily_dialog", split="train[:1000]")

## Preparaci?n de texto y tokenizaci?n

In [ ]:
def format_dialog(example):
    return {"text": " ".join(example["dialog"])}

text_dataset = raw_dataset.map(format_dialog, remove_columns=raw_dataset.column_names)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_dataset = text_dataset.map(tokenize, batched=True, remove_columns=text_dataset.column_names)

## Entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir="results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    save_strategy="epoch",
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
trainer.train()

## Guardado

In [ ]:
model.save_pretrained("trained_model")
tokenizer.save_pretrained("trained_model")